In [1]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../../")) ; from EPF import variables
sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/")) ; from memory_compute import *

# Detect free RAM now, cap this kernel so it can never exhaust the machine, and
# make every parquet read/write below stream in bounded batches instead of
# loading the whole frame at once.
install_memory_guard()

REGION        = variables.TARGET_REGION
ALL_REGIONS   = ["nsw", "qld", "vic", "sa"]
OTHER_REGIONS = [r for r in ALL_REGIONS if r != REGION]

# PREDISPATCH price forecast curve. Column predispatch_rrp_{region}_h{k} holds
# the forecast RRP for the k-th 30-min period strictly after each timestamp,
# taken from the most recent run available at that timestamp (merge_asof
# backward + per-horizon ffill in 1_Data_sources). h1 ~= +30min ... h78 ~= +39h.
# These are genuine ex-ante forecasts and are leakage-free by construction, so
# they are consumed as-is (no extra shifting).
PD_PRICE_PREFIX = "predispatch_rrp"
PD_HORIZONS     = 78
SPIKE_THRESHOLD = variables.SPIKE_THRESHOLD

SRC_PATH = "../1_Dataset/Processed_data/6_1_predispatch_price.parquet"
OUT_PATH = "../2_Features_build/Feature_data/6_1_predispatch_price.parquet"


[memory_guard] hard cap 13.3G virtual on this kernel (total RAM 14.8G, 9.1G free now). Runaway allocations fail cleanly; bounded streaming keeps normal work well under this.


In [2]:
# The final cell streams the source in bounded row-batches straight to disk.
# Here we only pull a tiny sample so the feature functions below can be previewed
# without ever holding the full frame in memory.
sample = peek_parquet(SRC_PATH, 10)
sample.iloc[:, :6]


,predispatch_rrp_nsw_h1,predispatch_rrp_nsw_h2,predispatch_rrp_nsw_h3,predispatch_rrp_nsw_h4,predispatch_rrp_nsw_h5,predispatch_rrp_nsw_h6
Date,,,,,,
2018-01-01 00:00:00,95.790001,89.607597,73.809998,69.634338,68.500000,65.000000
2018-01-01 00:05:00,95.790001,89.607597,73.809998,69.634338,68.500000,65.000000
2018-01-01 00:10:00,95.790001,89.607597,73.809998,69.634338,68.500000,65.000000
2018-01-01 00:15:00,95.790001,89.607597,73.809998,69.634338,68.500000,65.000000
2018-01-01 00:20:00,95.790001,89.607597,73.809998,69.634338,68.500000,65.000000
2018-01-01 00:25:00,95.790001,89.607597,73.809998,69.634338,68.500000,65.000000
2018-01-01 00:30:00,92.500000,74.923508,69.150429,68.500000,65.489998,66.100723
2018-01-01 00:35:00,92.500000,74.923508,69.150429,68.500000,65.489998,66.100723
2018-01-01 00:40:00,92.500000,74.923508,69.150429,68.500000,65.489998,66.100723


In [3]:
def _add_predispatch_price_curve(df: pd.DataFrame) -> pd.DataFrame:
    """
    Expose the target region's full predispatch RRP forecast curve as features.
    Each horizon k aligns to a future 30-min delivery period, so these forecasts
    are the single strongest predictor for the corresponding target horizon.
    They are ex-ante forecasts (leakage-free) and are used unshifted.
    Returns only the new columns to avoid copying the full base frame.
    """
    R = REGION
    new_cols = {}
    for k in range(1, PD_HORIZONS + 1):
        c = f"{PD_PRICE_PREFIX}_{R}_h{k}"
        if c in df.columns:
            new_cols[f"pd_price_{R}_fh{k}"] = df[c].astype(np.float32)
    return pd.DataFrame(new_cols, index=df.index)


_add_predispatch_price_curve(sample)[:10]


,pd_price_nsw_fh1,pd_price_nsw_fh2,pd_price_nsw_fh3,pd_price_nsw_fh4,pd_price_nsw_fh5,pd_price_nsw_fh6,pd_price_nsw_fh7,pd_price_nsw_fh8,pd_price_nsw_fh9,pd_price_nsw_fh10,...,pd_price_nsw_fh69,pd_price_nsw_fh70,pd_price_nsw_fh71,pd_price_nsw_fh72,pd_price_nsw_fh73,pd_price_nsw_fh74,pd_price_nsw_fh75,pd_price_nsw_fh76,pd_price_nsw_fh77,pd_price_nsw_fh78
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:00:00,95.790001,89.607597,73.809998,69.634338,68.500000,65.000000,65.000000,64.328148,55.010700,55.010281,...,55.010139,55.009998,55.009998,55.009998,55.009998,55.009998,55.009998,54.98,54.98,54.98
2018-01-01 00:05:00,95.790001,89.607597,73.809998,69.634338,68.500000,65.000000,65.000000,64.328148,55.010700,55.010281,...,55.010139,55.009998,55.009998,55.009998,55.009998,55.009998,55.009998,54.98,54.98,54.98
2018-01-01 00:10:00,95.790001,89.607597,73.809998,69.634338,68.500000,65.000000,65.000000,64.328148,55.010700,55.010281,...,55.010139,55.009998,55.009998,55.009998,55.009998,55.009998,55.009998,54.98,54.98,54.98
2018-01-01 00:15:00,95.790001,89.607597,73.809998,69.634338,68.500000,65.000000,65.000000,64.328148,55.010700,55.010281,...,55.010139,55.009998,55.009998,55.009998,55.009998,55.009998,55.009998,54.98,54.98,54.98
2018-01-01 00:20:00,95.790001,89.607597,73.809998,69.634338,68.500000,65.000000,65.000000,64.328148,55.010700,55.010281,...,55.010139,55.009998,55.009998,55.009998,55.009998,55.009998,55.009998,54.98,54.98,54.98
2018-01-01 00:25:00,95.790001,89.607597,73.809998,69.634338,68.500000,65.000000,65.000000,64.328148,55.010700,55.010281,...,55.010139,55.009998,55.009998,55.009998,55.009998,55.009998,55.009998,54.98,54.98,54.98
2018-01-01 00:30:00,92.500000,74.923508,69.150429,68.500000,65.489998,66.100723,64.316658,56.765442,55.010281,55.010281,...,55.010139,55.009998,55.009998,55.009998,55.009998,55.009998,55.009998,54.98,54.98,54.98
2018-01-01 00:35:00,92.500000,74.923508,69.150429,68.500000,65.489998,66.100723,64.316658,56.765442,55.010281,55.010281,...,55.010139,55.009998,55.009998,55.009998,55.009998,55.009998,55.009998,54.98,54.98,54.98
2018-01-01 00:40:00,92.500000,74.923508,69.150429,68.500000,65.489998,66.100723,64.316658,56.765442,55.010281,55.010281,...,55.010139,55.009998,55.009998,55.009998,55.009998,55.009998,55.009998,54.98,54.98,54.98


In [4]:
def _add_predispatch_price_shape(df: pd.DataFrame) -> pd.DataFrame:
    """
    Shape statistics of the target region's forecast price curve: expected
    level, dispersion and forecast spike counts over the next 6h / 24h / 39h,
    plus the slope from the near to the far end of the curve. These summarise
    where and how hard the market expects prices to move. Leakage-free.
    Returns only the new columns to avoid copying the full base frame.
    """
    R = REGION
    new_cols = {}
    windows = [("6h", range(1, 13)), ("24h", range(1, 49)), ("39h", range(1, PD_HORIZONS + 1))]
    for lab, ks in windows:
        cc = [f"{PD_PRICE_PREFIX}_{R}_h{k}" for k in ks if f"{PD_PRICE_PREFIX}_{R}_h{k}" in df.columns]
        sub = df[cc]
        new_cols[f"pd_price_{R}_fmean_{lab}"]   = sub.mean(axis=1).astype(np.float32)
        new_cols[f"pd_price_{R}_fmax_{lab}"]    = sub.max(axis=1).astype(np.float32)
        new_cols[f"pd_price_{R}_fstd_{lab}"]    = sub.std(axis=1).astype(np.float32)
        new_cols[f"pd_price_{R}_fspikes_{lab}"] = (sub > SPIKE_THRESHOLD).sum(axis=1).astype(np.float32)

    early = df[[f"{PD_PRICE_PREFIX}_{R}_h{k}" for k in range(1, 7)]].mean(axis=1)
    late  = df[[f"{PD_PRICE_PREFIX}_{R}_h{k}" for k in range(PD_HORIZONS - 5, PD_HORIZONS + 1)]].mean(axis=1)
    new_cols[f"pd_price_{R}_fslope"] = (late - early).astype(np.float32)

    return pd.DataFrame(new_cols, index=df.index)


_add_predispatch_price_shape(sample)[:10]


,pd_price_nsw_fmean_6h,pd_price_nsw_fmax_6h,pd_price_nsw_fstd_6h,pd_price_nsw_fspikes_6h,pd_price_nsw_fmean_24h,pd_price_nsw_fmax_24h,pd_price_nsw_fstd_24h,pd_price_nsw_fspikes_24h,pd_price_nsw_fmean_39h,pd_price_nsw_fmax_39h,pd_price_nsw_fstd_39h,pd_price_nsw_fspikes_39h,pd_price_nsw_fslope
Date,,,,,,,,,,,,,
2018-01-01 00:00:00,67.642624,95.790001,13.418847,0.0,66.017197,95.790001,10.901060,0.0,62.448692,95.790001,10.042187,0.0,-22.061985
2018-01-01 00:05:00,67.642624,95.790001,13.418847,0.0,66.017197,95.790001,10.901060,0.0,62.448692,95.790001,10.042187,0.0,-22.061985
2018-01-01 00:10:00,67.642624,95.790001,13.418847,0.0,66.017197,95.790001,10.901060,0.0,62.448692,95.790001,10.042187,0.0,-22.061985
2018-01-01 00:15:00,67.642624,95.790001,13.418847,0.0,66.017197,95.790001,10.901060,0.0,62.448692,95.790001,10.042187,0.0,-22.061985
2018-01-01 00:20:00,67.642624,95.790001,13.418847,0.0,66.017197,95.790001,10.901060,0.0,62.448692,95.790001,10.042187,0.0,-22.061985
2018-01-01 00:25:00,67.642624,95.790001,13.418847,0.0,66.017197,95.790001,10.901060,0.0,62.448692,95.790001,10.042187,0.0,-22.061985
2018-01-01 00:30:00,64.815651,92.500000,11.089302,0.0,65.773483,92.500000,10.178354,0.0,62.157696,92.500000,9.537688,0.0,-17.782444
2018-01-01 00:35:00,64.815651,92.500000,11.089302,0.0,65.773483,92.500000,10.178354,0.0,62.157696,92.500000,9.537688,0.0,-17.782444
2018-01-01 00:40:00,64.815651,92.500000,11.089302,0.0,65.773483,92.500000,10.178354,0.0,62.157696,92.500000,9.537688,0.0,-17.782444


In [5]:
def _add_predispatch_neighbour_price(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compact forecast-price context for the other regions (expected level and
    peak over the next 24h). Interconnected regions' forecast prices bound the
    import/export price into the target region. Leakage-free forecasts.
    Returns only the new columns to avoid copying the full base frame.
    """
    new_cols = {}
    for r in OTHER_REGIONS:
        cc = [f"{PD_PRICE_PREFIX}_{r}_h{k}" for k in range(1, 49) if f"{PD_PRICE_PREFIX}_{r}_h{k}" in df.columns]
        if cc:
            sub = df[cc]
            new_cols[f"pd_price_{r}_fmean_24h"] = sub.mean(axis=1).astype(np.float32)
            new_cols[f"pd_price_{r}_fmax_24h"]  = sub.max(axis=1).astype(np.float32)
    return pd.DataFrame(new_cols, index=df.index)


_add_predispatch_neighbour_price(sample)[:10]


,pd_price_qld_fmean_24h,pd_price_qld_fmax_24h,pd_price_vic_fmean_24h,pd_price_vic_fmax_24h,pd_price_sa_fmean_24h,pd_price_sa_fmax_24h
Date,,,,,,
2018-01-01 00:00:00,65.593758,87.842361,63.101486,99.399323,71.071434,116.642769
2018-01-01 00:05:00,65.593758,87.842361,63.101486,99.399323,71.071434,116.642769
2018-01-01 00:10:00,65.593758,87.842361,63.101486,99.399323,71.071434,116.642769
2018-01-01 00:15:00,65.593758,87.842361,63.101486,99.399323,71.071434,116.642769
2018-01-01 00:20:00,65.593758,87.842361,63.101486,99.399323,71.071434,116.642769
2018-01-01 00:25:00,65.593758,87.842361,63.101486,99.399323,71.071434,116.642769
2018-01-01 00:30:00,65.478302,87.835419,62.855160,91.096184,70.717133,105.000053
2018-01-01 00:35:00,65.478302,87.835419,62.855160,91.096184,70.717133,105.000053
2018-01-01 00:40:00,65.478302,87.835419,62.855160,91.096184,70.717133,105.000053


In [6]:
def add_predispatch_features(df: pd.DataFrame) -> pd.DataFrame:
    # All three groups are row-wise over the horizon columns, so they compute
    # identically on a row-batch as on the full frame.
    return pd.concat(
        [
            _add_predispatch_price_curve(df),
            _add_predispatch_price_shape(df),
            _add_predispatch_neighbour_price(df),
        ],
        axis=1,
    )


# Retain core columns (keep_source=True): the predispatch RRP forecast curve is
# an ex-ante forecast from the run available at t -> leakage-free by
# construction, used unshifted. Streams source + new features to disk in bounded
# batches; peak RAM is one batch, so this cannot exhaust memory.
stream_transform_parquet(SRC_PATH, OUT_PATH, transform=add_predispatch_features, keep_source=True)

import pyarrow.parquet as pq
meta = pq.ParquetFile(OUT_PATH).metadata
print("Total features:", meta.num_columns)
(meta.num_rows, meta.num_columns)


[stream] 6_1_predispatch_price.parquet: 893,665 rows x 391 cols, largest row group 0.56G -> column-block strategy (bounded regardless of free RAM)


Merge rows: 100%|██████████| 21/21 [00:15<00:00,  1.38batch/s]

[stream] wrote 893,665 rows -> ../2_Features_build/Feature_data/6_1_predispatch_price.parquet
Total features: 488


(893665, 488)

In [7]:
# Free this kernel's memory so the next notebook has RAM to work with
# (clears data variables + returns freed heap to the OS).
release_memory()


[release_memory] cleared 15 variable(s); kernel rss 0.42G, 9.1G RAM free now
